# 华南理工大学五山校区周边餐饮分析**高德POI采集 + DeepSeek智能分类 + 交互式可视化**---## 项目流程1. 高德 5×5 网格采集（25 个子区域，2000m 半径）→ 原始数据2. DeepSeek 批量分类（每 10 条一批，结果缓存）→ 带类别标签数据3. Folium 地图（LayerControl + MarkerCluster + HeatMap + 图例面板）4. Matplotlib 柱状图/饼图 + 分析总结> 运行前请在 `.env` 文件中填写 `AMAP_KEY` 和 `DEEPSEEK_API_KEY`

## 环境设置导入依赖模块，检查 API 配置。

In [ ]:
import pandas as pdimport warningswarnings.filterwarnings('ignore')from config import AMAP_KEY, DEEPSEEK_API_KEY, RAW_DATA_FILE, FINAL_DATA_FILEfrom poi_fetcher import collect_all_shops, save_raw_datafrom classifier import classify_allfrom visualizer import create_map, create_charts, print_summarydef check_keys():    ok = True    if not AMAP_KEY or AMAP_KEY == 'your_amap_key_here':        print('[警告] 高德地图 AMAP_KEY 未配置')        ok = False    if not DEEPSEEK_API_KEY or DEEPSEEK_API_KEY == 'your_deepseek_key_here':        print('[警告] DeepSeek DEEPSEEK_API_KEY 未配置')        ok = False    if ok:        print('API 密钥配置完成')    return okcheck_keys()

---## 步骤 1：高德API网格化采集以华南理工大学五山校区（113.351, 23.155）为中心，5×5 网格共 25 个子区域，每个 500m 半径，覆盖 ~2000m 范围。高德 `types=050000` 获取全品类餐饮 POI。遇到 `CUQPS_HAS_EXCEEDED_THE_LIMIT` 限流自动重试（最多 3 次，2s/4s/8s 指数退避）。

In [ ]:
# 采集所有餐饮POIdf_raw = collect_all_shops()# 查看前5条数据df_raw.head()

In [ ]:
# 数据概览print('数据形状:', df_raw.shape)print('列名:', list(df_raw.columns))df_raw.describe()

In [ ]:
# 保存原始数据save_raw_data(df_raw, RAW_DATA_FILE)

10 个餐饮类别：**火锅 | 烧烤 | 茶饮咖啡 | 粉面小吃 | 快餐简餐 | 中餐正餐 | 异国料理 | 面包甜点 | 早餐粥铺 | 其他**

含 13 条边界案例 few-shot 示例（如肠粉→早餐、螺蛳粉→粉面小吃、韩式烤肉→烧烤）。

In [ ]:
# 执行批量分类df = classify_all(df_raw)# 查看分类结果df[['name', 'address', 'category']].head(10)

In [ ]:
# 保存最终数据df.to_excel(FINAL_DATA_FILE, index=False, engine='openpyxl')print('已保存:', FINAL_DATA_FILE)

功能说明：
- **右上角图层控制器**：10 个类别 + 热力图，自由勾选开关
- **MarkerCluster**：邻近店铺自动聚合显示数量，点击展开
- **DivIcon 标记**：20px 彩色圆点，不同类别不同颜色
- **悬停提示**：鼠标划过显示店名
- **点击弹窗**：店名 + 类别 + 地址 + 高德导航链接
- **左侧图例面板**：店铺总数 + 各类数量/占比 + 采集日期
- **左上角全屏按钮** + **右下角鼠标坐标**

In [ ]:
# 生成交互式地图（Notebook 中可直接显示）m = create_map(df)m  # 显示地图

---## 步骤 4：统计图表类别数量柱状图 + 占比饼图。自动检测中文字体（SimHei / 微软雅黑）。

In [ ]:
create_charts(df)

In [ ]:
# 在 Notebook 中直接显示图表from IPython.display import Image, displaydisplay(Image(filename='category_bar.png'))display(Image(filename='category_pie.png'))

---## 分析总结

In [ ]:
print_summary(df)

---## 可扩展方向- 接入大众点评评分与人均消费- 时间段分析（早餐/夜宵店铺分布）- Streamlit / PyEcharts Dashboard- 店铺到校区各门的距离计算---**Author**：张阳**Date**：2026.06